# Shadow Revenue Detection System

# Silver Layer

## Objective

The Silver layer focuses on improving the quality of the raw datasets received from the Bronze layer. In this notebook, the data is profiled, cleaned, validated, standardized, and stored as Delta tables for downstream business analytics.

## Import Required Libraries

Import the PySpark libraries required for data profiling, cleaning, validation, and transformation.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

## Project Configuration

Define reusable configuration variables for accessing the Bronze and Silver layers.

In [0]:
CATALOG = "shadow_revenue_catalog"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

## Load Bronze Tables

Read the raw Delta tables created in the Bronze layer. These datasets will be profiled and cleaned before being stored in the Silver layer.

In [0]:
# Read Bronze Customers Table
customers_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers")

# Read Bronze Products Table
products_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.products")

# Read Bronze Orders Table
orders_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.orders")

# Read Bronze Payments Table
payments_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.payments")

## The Bad (Messy) Silver Layer
###  No transformations applied. Duplicates retained, orphan payments kept, no type safety.

In [0]:
# BAD PIPELINE
# No Cleaning
# No Deduplication
# No Type Casting

silver_customers_bad = customers_df
silver_products_bad = products_df.filter(col("is_current") == 1)
silver_orders_bad = orders_df
silver_payments_bad = payments_df

In [0]:
# register temp views
silver_customers_bad.createOrReplaceTempView("silver_customers_bad")
silver_products_bad.createOrReplaceTempView("silver_products_bad")
silver_orders_bad.createOrReplaceTempView("silver_orders_bad")
silver_payments_bad.createOrReplaceTempView("silver_payments_bad")

## save silver bad tables

In [0]:
silver_customers_bad.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.customers_bad")

silver_products_bad.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.products_bad")

silver_orders_bad.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.orders_bad")

silver_payments_bad.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.payments_bad")

## Data Quality Assessment

Before applying any transformations, assess the quality of the datasets. This helps identify missing values, duplicate records, and schema-related issues that need to be addressed.

In [0]:
# Display total records in each dataset

datasets = {
    "Customers": customers_df,
    "Products": products_df,
    "Orders": orders_df,
    "Payments": payments_df
}

for name, df in datasets.items():
    print(f"{name}: {df.count()} records")

Customers: 2000 records
Products: 200 records
Orders: 20400 records
Payments: 18600 records


In [0]:
# Check null values in each column

for name, df in datasets.items():

    print(f"\n{name} Dataset")

    df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).show()


Customers Dataset
+-----------+----+----+-----------+
|customer_id|name|city|signup_date|
+-----------+----+----+-----------+
|          0|   0|   0|          0|
+-----------+----+----+-----------+


Products Dataset
+----------+-----+--------+--------------+--------+----------+
|product_id|price|category|effective_date|end_date|is_current|
+----------+-----+--------+--------------+--------+----------+
|         0|    0|       0|             0|     100|         0|
+----------+-----+--------+--------------+--------+----------+


Orders Dataset
+--------+----------+-----------+--------+-----+----------+------------+-------+--------+
|order_id|product_id|customer_id|quantity|price|order_date|order_status|channel|discount|
+--------+----------+-----------+--------+-----+----------+------------+-------+--------+
|       0|         0|          0|       0|    0|         0|           0|      0|       0|
+--------+----------+-----------+--------+-----+----------+------------+-------+--------+


In [0]:
# Check duplicate records

for name, df in datasets.items():

    duplicates = df.count() - df.dropDuplicates().count()

    print(f"{name}: {duplicates} duplicate records")

Customers: 0 duplicate records
Products: 0 duplicate records
Orders: 400 duplicate records
Payments: 0 duplicate records


## Data Cleaning

Remove duplicate records and standardize the datasets while preserving only valid records for downstream processing.

## SILVER GOOD PIPELINE

In [0]:
# Remove Duplicate Orders
# Keep latest record for each order_id

window_spec = (
    Window
    .partitionBy("order_id")
    .orderBy(col("order_date").desc())
)

orders_clean = (
    orders_df
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
# Convert Financial Columns
orders_clean = (
    orders_clean
    .withColumn("price", col("price").cast(DecimalType(10,4)))
    .withColumn("order_date", to_date("order_date"))
)

payments_clean = (
    payments_df
    .withColumn("payment_amount",
                col("payment_amount").cast(DecimalType(10,4)))
    .withColumn("payment_date",
                to_date("payment_date"))
)

products_clean = (
    products_df
    .withColumn("price",
                col("price").cast(DecimalType(10,4)))
    .withColumn("effective_date",
                to_date("effective_date"))
    .withColumn("end_date",
                to_date("end_date"))
)

In [0]:
# Remove NULL Primary Keys
orders_clean = (
    orders_clean
    .filter(col("order_id").isNotNull())
)

payments_clean = (
    payments_clean
    .filter(col("order_id").isNotNull())
)

customers_clean = (
    customers_df
    .filter(col("customer_id").isNotNull())
)

products_clean = (
    products_clean
    .filter(col("product_id").isNotNull())
)

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

# 1. Generate the surrogate payment ID
payments_clean = payments_df.withColumn(
    "payment_id",
    monotonically_increasing_id() + 1
)

# 2. Reorder to place 'payment_id' as the first column
payments_clean = payments_clean.select("payment_id", *[col for col in payments_df.columns])


In [0]:
# Remove rows where primary key is null

customers_clean = customers_clean.filter(col("customer_id").isNotNull())

products_clean = products_clean.filter(col("product_id").isNotNull())

orders_clean = orders_clean.filter(col("order_id").isNotNull())

payments_clean = payments_clean.filter(col("payment_id").isNotNull())

display(customers_clean.count()) 
displayHTML("<br>")

display(products_clean.count())
displayHTML("<br>")

display(orders_clean.count())
displayHTML("<br>")

display(payments_clean.count())


## Business Validation

Validate the datasets against basic business rules to ensure that invalid records are excluded from downstream processing.

In [0]:
# Product price should be greater than zero

products_clean = products_clean.filter(col("price") > 0)
# Order quantity should be greater than zero

orders_clean = orders_clean.filter(col("quantity") > 0)

# Payment amount should be greater than zero

payments_clean = payments_clean.filter(col("payment_amount") > 0)

In [0]:
customers_clean.createOrReplaceTempView("silver_customers_good")

products_clean.createOrReplaceTempView("silver_products_good")

orders_clean.createOrReplaceTempView("silver_orders_good")

payments_clean.createOrReplaceTempView("silver_payments_good")

## Store Silver Tables

Store the cleaned datasets as Delta tables in the Silver schema.

In [0]:
customers_clean.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.customers_good")

products_clean.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.products_good")

orders_clean.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.orders_good")

payments_clean.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.payments_good")